# Dense Scaling Laws for Fixed-Width Addition

This notebook analyzes the completed dense baseline:

- **5 model sizes**
- **8 independent training budgets**
- **3 seeds per point**
- **120 verified Colab T4 runs**

The analysis reports measured three-seed statistics, tests whether an additive Chinchilla-style law is identifiable, and always provides the empirical compute-loss frontier.

## Definitions

\(N\) is trainable parameters. \(D\) is full sequence tokens processed, including repeated exposure to examples from a fixed pool of 200,000 unique addition pairs.

\[
C \approx 6ND
\]

This is task-specific exposure scaling, not a universal internet-scale Chinchilla law.

In [ ]:
from pathlib import Path
import csv
import json
import os
import subprocess
import sys

REPOSITORY = "https://github.com/marcoharuni/jax-addition-transformer.git"
BRANCH = "dense-scaling"
REPO_DIR = Path("/content/jax-addition-transformer")

if "COLAB_RELEASE_TAG" in os.environ:
    if not REPO_DIR.exists():
        subprocess.run(
            ["git", "clone", "--branch", BRANCH, "--single-branch", REPOSITORY, str(REPO_DIR)],
            check=True,
        )
    os.chdir(REPO_DIR)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"],
        check=True,
    )
elif not (Path.cwd() / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repository root.")

print("Repository:", Path.cwd())

## 1. Verify and aggregate all 120 runs

In [ ]:
subprocess.run(
    [sys.executable, "experiments/dense_scaling/analyze_final.py"],
    check=True,
)

artifact = Path("artifacts/dense-scaling-final-t4")
summary = json.loads((artifact / "final_summary.json").read_text())
print(json.dumps(summary, indent=2))

## 2. Inspect the exact-match transition

In [ ]:
with (artifact / "transition_table.csv").open(newline="") as handle:
    transitions = list(csv.DictReader(handle))

for row in transitions:
    print(row)

## 3. Test scaling-law identifiability

The target surface is

\[
L(N,D)=E+A(N/10^6)^{-\alpha}+B(D/10^6)^{-\beta}.
\]

The fit is repeated at several minimum-loss cutoffs. Compute-optimal exponents are reported only when multiple cutoffs produce stable and consistent estimates.

In [ ]:
subprocess.run(
    [sys.executable, "experiments/dense_scaling/fit_final_scaling_law.py"],
    check=True,
)

fit = json.loads((artifact / "final_fit.json").read_text())
print(json.dumps(fit, indent=2))

## 4. Generate the final figures

In [ ]:
subprocess.run(
    [sys.executable, "experiments/dense_scaling/plot_final_results.py"],
    check=True,
)

from IPython.display import SVG, display

for figure_path in (
    "assets/dense_scaling_final_loss.svg",
    "assets/dense_scaling_final_exact_match.svg",
    "assets/dense_scaling_final_compute.svg",
    "assets/dense_scaling_final_frontier.svg",
):
    print(figure_path)
    display(SVG(filename=figure_path))

## 5. Inspect all 40 three-seed aggregates

In [ ]:
with (artifact / "final_grouped.csv").open(newline="") as handle:
    grouped = list(csv.DictReader(handle))

for row in grouped:
    print(
        f"{row['model']:<14} "
        f"steps={int(float(row['steps'])):>3} "
        f"N={int(float(row['parameter_count'])):>10,} "
        f"loss={float(row['validation_loss_mean']):.8f}"
        f"±{float(row['validation_loss_std']):.8f} "
        f"EM={float(row['greedy_exact_match_mean']):.5f}"
        f"±{float(row['greedy_exact_match_std']):.5f}"
    )

## 6. Reproduce training only when needed

The repository contains verified compact artifacts. Training is disabled by default because the full experiment contains 120 independent runs.

In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    import jax

    print("Backend:", jax.default_backend())
    print("Devices:", jax.devices())
    assert jax.default_backend() == "gpu"

    subprocess.run(
        [
            sys.executable,
            "experiments/dense_scaling/run_final_grid.py",
            "--continue-on-error",
        ],
        check=True,
    )
else:
    print("Training skipped. Set RUN_TRAINING = True on a GPU runtime to reproduce the grid.")

## 7. Dense baseline conclusion

This experiment freezes the dense reference for the MoE stage. The MoE study must preserve the same task, splits, token accounting, seeds, evaluation protocol, and explicit compute accounting.